# SE446 Milestone 2 - Chicago Crime Analytics with Spark + MLlib

Team Members:
- Mishari Al Mogren
- Abdulaziz Alnemer
- Ibrahim Alhagbani
- Nawaf Alshuaibi
- Abdulaziz Albaz




## Setup
**Author: Mishari Al Mogren**


In [ ]:
# ============================================
# Setup: Spark Session and Data Loading
# Author: Mishari Al Mogren
# ============================================

import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, hour, lower, to_timestamp, when

DATA_PATH = os.environ.get("CHICAGO_CRIME_DATA", "data/generated_10000_crimes.csv")

spark = SparkSession.builder \
    .appName("SE446_M2_Chicago_Crime") \
    .master(os.environ.get("SPARK_MASTER", "local[*]")) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Data path: {DATA_PATH}")


In [ ]:
# ============================================
# Setup: Read CSV and clean useful columns
# Author: Mishari Al Mogren
# ============================================

raw_df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

df = raw_df.withColumn("Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))) \
    .withColumn("District", col("District").cast("double")) \
    .withColumn("Domestic_str", lower(col("Domestic").cast("string"))) \
    .withColumn("label", when(lower(col("Arrest").cast("string")) == "true", 1).otherwise(0))

df = df.dropna(subset=["Primary Type", "Location Description", "Year", "District", "Hour", "Domestic_str", "label"])
df.cache()

print(f"Rows after cleaning: {df.count()}")
df.printSchema()
df.show(5, truncate=False)


## Phase A - Spark DataFrame Analytics


### Task 1: Crime Type Distribution
**Author: Abdulaziz Albaz**


In [ ]:
# ============================================
# Task 1: Crime Type Distribution using DataFrame
# Author: Abdulaziz Albaz
# ============================================

spark_task1 = df.groupBy("Primary Type").count().orderBy(col("count").desc())
print("Top 10 crime types using Spark DataFrame:")
spark_task1.show(10, truncate=False)

print("M1 MapReduce sample top crimes from README:")
print("THEFT 162688, BATTERY 151930, CRIMINAL DAMAGE 91241, NARCOTICS 74127, ASSAULT 54070")


### Task 2: Location Hotspots
**Author: Abdulaziz Alnemer**


In [ ]:
# ============================================
# Task 2: Location Hotspots using Spark SQL
# Author: Abdulaziz Alnemer
# ============================================

df.createOrReplaceTempView("crimes")

location_hotspots = spark.sql("""
    SELECT `Location Description`, COUNT(*) AS total
    FROM crimes
    GROUP BY `Location Description`
    ORDER BY total DESC
    LIMIT 10
""")

location_hotspots.show(truncate=False)

print("M1 MapReduce sample top locations from README:")
print("STREET 245437, RESIDENCE 136238, APARTMENT 60925, SIDEWALK 47407, OTHER 29213")


### Task 3: Crime Trend Over Years
**Author: Ibrahim Alhagbani**


In [ ]:
# ============================================
# Task 3: Crime Trend Over Years with simple chart
# Author: Ibrahim Alhagbani
# ============================================

yearly = df.groupBy("Year").count().orderBy("Year")
yearly.show(50)

try:
    import matplotlib.pyplot as plt
    os.makedirs("output", exist_ok=True)
    yearly_pdf = yearly.toPandas()
    plt.figure(figsize=(8, 4))
    plt.plot(yearly_pdf["Year"], yearly_pdf["count"], marker="o")
    plt.title("Chicago Crime Count by Year")
    plt.xlabel("Year")
    plt.ylabel("Crime Count")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("output/year_trend.png")
    plt.show()
    print("Chart saved to output/year_trend.png")
except Exception as e:
    print("Chart skipped. Table above is enough for cluster mode.")
    print(e)


### Task 4: Arrest Rate Analysis
**Author: Nawaf Alshuaibi**


In [ ]:
# ============================================
# Task 4: Arrest Rate Analysis
# Author: Nawaf Alshuaibi
# ============================================

total = df.count()
arrests = df.filter(col("label") == 1).count()
print(f"Overall arrest rate: {arrests / total:.2%} ({arrests}/{total})")

arrest_by_type = df.groupBy("Primary Type") \
    .agg(count("*").alias("total"), avg("label").alias("arrest_rate")) \
    .orderBy(col("arrest_rate").desc())

print("Highest arrest rates by crime type:")
arrest_by_type.show(10, truncate=False)

print("Lowest arrest rates by crime type:")
arrest_by_type.orderBy(col("arrest_rate").asc()).show(10, truncate=False)


## Phase B - Spark MLlib Arrest Prediction


### Task 5: Feature Engineering Pipeline
**Author: Ibrahim Alhagbani**


In [ ]:
# ============================================
# Task 5: Feature Engineering Pipeline
# Author: Ibrahim Alhagbani
# ============================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

feature_cols = ["District", "crime_index", "Hour", "domestic_index"]

crime_indexer = StringIndexer(inputCol="Primary Type", outputCol="crime_index", handleInvalid="skip")
domestic_indexer = StringIndexer(inputCol="Domestic_str", outputCol="domestic_index", handleInvalid="skip")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

feature_pipeline = Pipeline(stages=[crime_indexer, domestic_indexer, assembler])
feature_model = feature_pipeline.fit(df)
feature_df = feature_model.transform(df)

feature_df.select("Primary Type", "District", "Hour", "Domestic_str", "features", "label").show(5, truncate=False)
print("Feature vector positions: [District, crime_index, Hour, domestic_index]")

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()
print(f"Training rows: {train_df.count()}")
print(f"Testing rows: {test_df.count()}")


### Task 6: Train and Evaluate Three Models
**Author: Abdulaziz Albaz**


In [ ]:
# ============================================
# Task 6: Train and Evaluate Three Models
# Author: Abdulaziz Albaz
# ============================================

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

binary_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
multi_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

def get_confusion(predictions):
    rows = predictions.groupBy("label", "prediction").count().collect()
    d = {(int(r["label"]), int(r["prediction"])): r["count"] for r in rows}
    return d.get((0,0), 0), d.get((0,1), 0), d.get((1,0), 0), d.get((1,1), 0)

def train_and_score(name, classifier):
    pipeline = Pipeline(stages=[crime_indexer, domestic_indexer, assembler, classifier])
    start = time.time()
    model = pipeline.fit(train_df)
    train_time = time.time() - start
    predictions = model.transform(test_df)
    tn, fp, fn, tp = get_confusion(predictions)
    return model, {
        "Model": name,
        "AUC": binary_eval.evaluate(predictions),
        "Accuracy": multi_eval.evaluate(predictions, {multi_eval.metricName: "accuracy"}),
        "F1": multi_eval.evaluate(predictions, {multi_eval.metricName: "f1"}),
        "Precision": multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedPrecision"}),
        "Recall": multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedRecall"}),
        "Training Time": train_time,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

models = [
    ("Logistic Regression", LogisticRegression(featuresCol="features", labelCol="label", maxIter=100, regParam=0.01)),
    ("Random Forest", RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=5, seed=42)),
    ("GBT", GBTClassifier(featuresCol="features", labelCol="label", maxIter=50, maxDepth=5, seed=42)),
]

trained = {}
results = []
for name, clf in models:
    print(f"Training {name}...")
    model, row = train_and_score(name, clf)
    trained[name] = model
    results.append(row)

print("Model comparison:")
for r in results:
    print(r)


### Task 7: Feature Importances and Interpretation
**Author: Nawaf Alshuaibi**


In [ ]:
# ============================================
# Task 7: Feature Importances and Interpretation
# Author: Nawaf Alshuaibi
# ============================================

rf_model = trained["Random Forest"].stages[-1]
importances = list(rf_model.featureImportances)

print("Random Forest feature importances:")
for name, value in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
    print(f"{name:<16} {value:.4f} " + "#" * int(value * 40))

best_model = max(results, key=lambda r: r["AUC"])
print(f"Best model by AUC: {best_model['Model']} with AUC={best_model['AUC']:.3f}")
print("Interpretation: crime type and domestic status usually matter because arrest probability differs a lot by category. Tree models can split categories better than Logistic Regression when categories are only encoded as indexes.")


## Phase C - Deployment Notes
**Author: Mishari Al Mogren**

Task 9 local execution was completed in Colab using `local[*]` mode and the generated 10,000-row dataset. Evidence is saved in the output folder.

Task 10 cluster client mode command:
```bash
spark-submit \
  --master yarn \
  --deploy-mode client \
  --conf spark.yarn.am.memory=256m \
  --conf spark.yarn.am.memoryOverhead=128m \
  --executor-memory 512m \
  --driver-memory 512m \
  task10_mishari_cluster_client.py \
  hdfs:///data/chicago_crimes.csv \
  2>&1 | tee output/cluster_client/task10_client_run.log
```

Task 11 spark-submit cluster mode command:
```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --driver-memory 465m \
  --num-executors 1 \
  --executor-memory 768m \
  --executor-cores 1 \
  --conf spark.driver.memoryOverhead=32m \
  --conf spark.yarn.am.memory=465m \
  --conf spark.yarn.am.memoryOverhead=32m \
  --conf spark.driver.maxResultSize=128m \
  --conf spark.yarn.appMasterEnv.PYSPARK_PYTHON=python3.12 \
  --conf spark.executorEnv.PYSPARK_PYTHON=python3.12 \
  m2_spark_ml.py hdfs:///data/chicago_crimes.csv \
  2>&1 | tee output/spark_submit/task11_submit_terminal.log
```

Task 11 YARN log collection:
```bash
yarn logs -applicationId <application_id> > output/spark_submit/run.log
wc -l output/spark_submit/run.log
head -100 output/spark_submit/run.log
```


In [ ]:
# Cleanup when done
train_df.unpersist()
test_df.unpersist()
df.unpersist()
# spark.stop()
